In [10]:
"""
Image → PDF → Voice Automation Tool
Requirements: pip install gradio pypdf reportlab Pillow pyttsx3 pdfplumber gtts
"""
!pip install pdfplumber 
!pip install gTTS
!pip install pdfplumber gTTS pypdf reportlab

import os
import tempfile
import gradio as gr
from PIL import Image
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
import pdfplumber
from gtts import gTTS
from pypdf import PdfReader


# ─────────────────────────────────────────────
# BACKEND FUNCTIONS
# ─────────────────────────────────────────────

def images_to_pdf(image_files):
    if not image_files:
        return None, "⚠️ Please upload at least one image."
    out_path = os.path.join(tempfile.gettempdir(), "converted_images.pdf")
    page_w, page_h = A4
    c = canvas.Canvas(out_path, pagesize=A4)
    for img_file in image_files:
        img = Image.open(img_file)
        img_w, img_h = img.size
        margin = 20
        max_w, max_h = page_w - 2 * margin, page_h - 2 * margin
        ratio = min(max_w / img_w, max_h / img_h)
        draw_w, draw_h = img_w * ratio, img_h * ratio
        x, y = (page_w - draw_w) / 2, (page_h - draw_h) / 2
        c.drawImage(img_file, x, y, width=draw_w, height=draw_h, preserveAspectRatio=True)
        c.showPage()
    c.save()
    return out_path, f"✅ PDF created with {len(image_files)} page(s)."


def extract_text_from_pdf(pdf_file):
    if pdf_file is None:
        return "", "⚠️ Please upload a PDF."
    text_chunks = []
    try:
        with pdfplumber.open(pdf_file) as pdf:
            for i, page in enumerate(pdf.pages):
                t = page.extract_text()
                if t:
                    text_chunks.append(f"[Page {i+1}]\n{t}")
    except Exception:
        reader = PdfReader(pdf_file)
        for i, page in enumerate(reader.pages):
            t = page.extract_text()
            if t:
                text_chunks.append(f"[Page {i+1}]\n{t}")
    if not text_chunks:
        return "", "⚠️ No readable text found — PDF may be image-only."
    return "\n\n".join(text_chunks), f"✅ Extracted text from {len(text_chunks)} page(s)."


def text_to_voice(text, lang="en", slow=False):
    if not text or not text.strip():
        return None, "⚠️ No text to convert."
    out_path = os.path.join(tempfile.gettempdir(), "output_voice.mp3")
    tts = gTTS(text=text.strip(), lang=lang, slow=slow)
    tts.save(out_path)
    return out_path, "✅ Voice file ready for download."


def pdf_to_voice_pipeline(pdf_file, lang="en", slow=False):
    text, msg = extract_text_from_pdf(pdf_file)
    if not text:
        return None, None, msg
    audio_path, voice_msg = text_to_voice(text, lang=lang, slow=slow)
    return text, audio_path, f"{msg}  •  {voice_msg}"


# ─────────────────────────────────────────────
# LANGUAGE OPTIONS
# ─────────────────────────────────────────────
LANGUAGE_CHOICES = [
    ("🇺🇸 English",  "en"),
    ("🇮🇳 Hindi",    "hi"),
    ("🇮🇳 Kannada",  "kn"),
    ("🇮🇳 Tamil",    "ta"),
    ("🇮🇳 Telugu",   "te"),
    ("🇫🇷 French",   "fr"),
    ("🇪🇸 Spanish",  "es"),
    ("🇩🇪 German",   "de"),
    ("🇸🇦 Arabic",   "ar"),
    ("🇯🇵 Japanese", "ja"),
]


# ─────────────────────────────────────────────
# MODERN MOBILE-FIRST CSS
# ─────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');

:root {
    --bg:         #0d0f14;
    --surface:    #161a23;
    --surface2:   #1e2330;
    --border:     #2a3040;
    --accent:     #6c63ff;
    --accent2:    #ff6584;
    --accent3:    #43e8b0;
    --text:       #e8ecf4;
    --muted:      #7a8499;
    --radius:     18px;
    --radius-sm:  10px;
    --font:       'Plus Jakarta Sans', sans-serif;
    --shadow:     0 8px 32px rgba(0,0,0,0.45);
}

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container {
    background: var(--bg) !important;
    font-family: var(--font) !important;
    color: var(--text) !important;
    margin: 0 !important;
}

.gradio-container {
    max-width: 800px !important;
    margin: 0 auto !important;
    min-height: 100vh;
}

/* ─── Header ─── */
#app-header {
    background: linear-gradient(160deg, #1a1d2e 0%, #0d0f14 100%);
    padding: 32px 20px 22px;
    text-align: center;
    border-bottom: 1px solid var(--border);
    position: relative;
    overflow: hidden;
}
#app-header::before {
    content:'';
    position:absolute;
    width:300px; height:300px;
    background: radial-gradient(circle, rgba(108,99,255,0.15) 0%, transparent 65%);
    top:-100px; left:50%; transform:translateX(-50%);
    border-radius:50%;
    pointer-events:none;
}
#app-header::after {
    content:'';
    position:absolute;
    width:180px; height:180px;
    background: radial-gradient(circle, rgba(67,232,176,0.10) 0%, transparent 65%);
    bottom:-60px; right:10%;
    border-radius:50%;
    pointer-events:none;
}
#app-logo {
    font-size:2.6rem;
    display:block;
    margin-bottom:6px;
    filter: drop-shadow(0 0 12px rgba(108,99,255,0.5));
}
#app-title { margin:0 !important; }
#app-title p, #app-title h1, #app-title h2 {
    font-size: 1.7rem !important;
    font-weight: 800 !important;
    letter-spacing: -0.5px;
    background: linear-gradient(90deg, #a78bfa 0%, #6c63ff 50%, #43e8b0 100%);
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    background-clip: text !important;
    margin: 0 !important;
    line-height: 1.2 !important;
}
#app-subtitle p {
    color: var(--muted) !important;
    font-size: 0.8rem !important;
    margin: 8px 0 0 !important;
    font-weight: 500;
    letter-spacing: 0.02em;
}

/* ─── Tab nav ─── */
.tab-nav {
    background: var(--surface) !important;
    border-bottom: 1px solid var(--border) !important;
    padding: 0 8px !important;
    display: flex !important;
    overflow-x: auto !important;
    scrollbar-width: none !important;
    gap: 0 !important;
}
.tab-nav::-webkit-scrollbar { display:none; }
.tab-nav button {
    font-family: var(--font) !important;
    font-size: 0.76rem !important;
    font-weight: 600 !important;
    color: var(--muted) !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 3px solid transparent !important;
    padding: 14px 13px 11px !important;
    border-radius: 0 !important;
    white-space: nowrap !important;
    transition: color 0.2s, border-color 0.2s !important;
    cursor: pointer !important;
    flex-shrink: 0 !important;
    letter-spacing: 0.01em !important;
}
.tab-nav button:hover { color: var(--text) !important; }
.tab-nav button.selected {
    color: var(--accent) !important;
    border-bottom-color: var(--accent) !important;
}

/* ─── Tab body ─── */
.tabitem {
    padding: 20px 16px 40px !important;
    background: var(--bg) !important;
}

/* ─── Section labels ─── */
.section-label span, .section-label p {
    font-size: 0.7rem !important;
    font-weight: 700 !important;
    letter-spacing: 0.09em !important;
    text-transform: uppercase !important;
    color: var(--muted) !important;
}

/* ─── Upload zone ─── */
.upload-zone .wrap {
    background: var(--surface2) !important;
    border: 2px dashed var(--border) !important;
    border-radius: var(--radius) !important;
    min-height: 110px !important;
    transition: border-color 0.2s, background 0.2s !important;
}
.upload-zone .wrap:hover {
    border-color: var(--accent) !important;
    background: rgba(108,99,255,0.06) !important;
}
.upload-zone label span {
    font-size: 0.8rem !important;
    font-weight: 600 !important;
    color: var(--muted) !important;
}

/* ─── Labels ─── */
label span, .gr-form > label > span {
    font-family: var(--font) !important;
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    color: var(--muted) !important;
}

/* ─── Textbox ─── */
textarea, input[type="text"] {
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius-sm) !important;
    color: var(--text) !important;
    font-family: var(--font) !important;
    font-size: 0.85rem !important;
    padding: 12px !important;
    transition: border-color 0.2s, box-shadow 0.2s !important;
    resize: none !important;
    line-height: 1.6 !important;
}
textarea:focus, input[type="text"]:focus {
    border-color: var(--accent) !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(108,99,255,0.15) !important;
}

/* ─── Dropdown ─── */
.svelte-select, select, .gr-dropdown > div {
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius-sm) !important;
    color: var(--text) !important;
    font-family: var(--font) !important;
    font-size: 0.84rem !important;
}

/* ─── Checkbox ─── */
.gr-checkbox-label span, input[type="checkbox"] + span {
    font-size: 0.82rem !important;
    color: var(--muted) !important;
}
input[type="checkbox"] { accent-color: var(--accent) !important; }

/* ─── PRIMARY BUTTON ─── */
button.primary, .gr-button-primary, button[variant="primary"] {
    font-family: var(--font) !important;
    font-size: 0.92rem !important;
    font-weight: 700 !important;
    letter-spacing: 0.02em !important;
    background: linear-gradient(135deg, #7c74ff 0%, #5a52e0 100%) !important;
    color: #fff !important;
    border: none !important;
    border-radius: var(--radius) !important;
    padding: 15px 24px !important;
    width: 100% !important;
    cursor: pointer !important;
    box-shadow: 0 4px 20px rgba(108,99,255,0.38) !important;
    transition: transform 0.15s, box-shadow 0.15s !important;
    margin-top: 4px !important;
}
button.primary:hover, .gr-button-primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(108,99,255,0.55) !important;
}
button.primary:active { transform: translateY(0) !important; }

/* ─── Status box ─── */
.status-pill textarea {
    background: rgba(67,232,176,0.07) !important;
    border: 1px solid rgba(67,232,176,0.22) !important;
    border-radius: var(--radius-sm) !important;
    color: var(--accent3) !important;
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    padding: 10px 12px !important;
    min-height: unset !important;
}

/* ─── Audio player ─── */
.gr-audio, [data-testid="audio"] {
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 10px !important;
}
audio { width:100% !important; accent-color: var(--accent) !important; }

/* ─── File download ─── */
.gr-file-preview, [data-testid="file"] {
    background: var(--surface2) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
}

/* ─── Divider ─── */
.divider { border:none !important; border-top:1px solid var(--border) !important; margin:16px 0 !important; }

/* ─── Feature chips ─── */
.chips-row {
    display:flex;
    flex-wrap:wrap;
    gap:8px;
    margin-bottom:16px;
}
.chip {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius:100px;
    padding: 5px 12px;
    font-size:0.73rem;
    font-weight:600;
    color: var(--muted);
    display:inline-flex;
    align-items:center;
    gap:5px;
}

/* ─── Footer ─── */
#footer {
    text-align:center;
    padding:16px;
    border-top:1px solid var(--border);
    background: var(--surface);
}
#footer p { color: var(--muted) !important; font-size:0.7rem !important; margin:0 !important; }

/* ─── Scrollbar ─── */
::-webkit-scrollbar { width:4px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius:4px; }

/* ─── Mobile ─── */
@media (max-width:500px) {
    .gradio-container { max-width:100% !important; }
    .tab-nav button { padding:12px 10px 9px !important; font-size:0.71rem !important; }
}
"""

# ─────────────────────────────────────────────
# BUILD UI
# ─────────────────────────────────────────────
with gr.Blocks(css=CSS, title="DocVoice · Convert & Listen", theme=gr.themes.Glass()) as demo:

    # ── Header ──────────────────────────────────
    with gr.Column(elem_id="app-header"):
        gr.HTML('<span id="app-logo">🎙️</span>')
        gr.Markdown("# AI-Based Smart PDF Reader and Voice Assistant", elem_id="app-title")
        gr.Markdown("Images → PDF &nbsp;·&nbsp; PDF → Voice &nbsp;·&nbsp; Text → Voice", elem_id="app-subtitle")

    # ── Feature chips ───────────────────────────
    gr.HTML("""
    <div style="padding:14px 16px 0; background:#0d0f14;">
      <div class="chips-row">
        <span class="chip">🖼️ PNG · JPG · WEBP</span>
        <span class="chip">📄 PDF extract</span>
        <span class="chip">🔊 MP3 download</span>
        <span class="chip">🌐 10 languages</span>
      </div>
    </div>
    """)

    with gr.Tabs(elem_id="main-tabs"):

        # ╔══════════════╗
        # ║  TAB 1       ║
        # ╚══════════════╝
        with gr.Tab("🖼️  Image → PDF"):

            gr.Markdown("Upload source", elem_classes="section-label")
            img_input = gr.File(
                label="Drop images here  (PNG / JPG / WEBP)",
                file_types=["image"],
                file_count="multiple",
                elem_classes="upload-zone",
            )

            img_btn = gr.Button("⚡  Convert to PDF", variant="primary")

            gr.HTML("<hr class='divider'>")
            gr.Markdown("Result", elem_classes="section-label")

            img_pdf_out = gr.File(label="📄 Download PDF")
            img_status  = gr.Textbox(label="Status", interactive=False, lines=1, elem_classes="status-pill")

            img_btn.click(fn=images_to_pdf, inputs=[img_input], outputs=[img_pdf_out, img_status])

        # ╔══════════════╗
        # ║  TAB 2       ║
        # ╚══════════════╝
        with gr.Tab("📄  PDF → Voice"):

            gr.Markdown("Upload PDF", elem_classes="section-label")
            pdf_input = gr.File(
                label="Drop your PDF here",
                file_types=[".pdf"],
                file_count="single",
                elem_classes="upload-zone",
            )

            gr.Markdown("Options", elem_classes="section-label")
            with gr.Row():
                lang_dropdown = gr.Dropdown(choices=LANGUAGE_CHOICES, value="en", label="🌐 Language")
                slow_toggle   = gr.Checkbox(label="🐢 Slow speech", value=False)

            pdf_voice_btn = gr.Button("🔊  Generate Voice", variant="primary")

            gr.HTML("<hr class='divider'>")
            gr.Markdown("Result", elem_classes="section-label")

            extracted_text = gr.Textbox(
                label="📝 Extracted Text",
                lines=7,
                interactive=False,
                placeholder="Extracted text will appear here…",
            )
            audio_player = gr.Audio(label="🔊 Listen & Download", type="filepath")
            pdf_status   = gr.Textbox(label="Status", interactive=False, lines=1, elem_classes="status-pill")

            pdf_voice_btn.click(
                fn=pdf_to_voice_pipeline,
                inputs=[pdf_input, lang_dropdown, slow_toggle],
                outputs=[extracted_text, audio_player, pdf_status],
            )

        # ╔══════════════╗
        # ║  TAB 3       ║
        # ╚══════════════╝
        with gr.Tab("✏️  Text → Voice"):

            gr.Markdown("Enter text", elem_classes="section-label")
            text_input = gr.Textbox(
                label="Your Text",
                lines=8,
                placeholder="Type or paste your text here…",
            )

            gr.Markdown("Options", elem_classes="section-label")
            with gr.Row():
                lang_dd2      = gr.Dropdown(choices=LANGUAGE_CHOICES, value="en", label="🌐 Language")
                slow_toggle2  = gr.Checkbox(label="🐢 Slow speech", value=False)

            text_btn = gr.Button("🎙️  Speak It", variant="primary")

            gr.HTML("<hr class='divider'>")
            gr.Markdown("Result", elem_classes="section-label")

            audio_out2  = gr.Audio(label="🔊 Listen & Download", type="filepath")
            text_status = gr.Textbox(label="Status", interactive=False, lines=1, elem_classes="status-pill")

            text_btn.click(
                fn=text_to_voice,
                inputs=[text_input, lang_dd2, slow_toggle2],
                outputs=[audio_out2, text_status],
            )

    # ── Footer ──────────────────────────────────
    gr.HTML("""
    <div id="footer">
      <p>DocVoice &nbsp;·&nbsp; Built with Gradio &nbsp;·&nbsp; gTTS · ReportLab · pdfplumber</p>
    </div>
    """)


if __name__ == "__main__":
    demo.launch(share=False, inbrowser=True)

C:\Users\Dell\AppData\Local\Temp\ipykernel_10520\3587959357.py:397: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="DocVoice · Convert & Listen", theme=gr.themes.Glass()) as demo:


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
